In [ ]:
import numpy as np
import pandas as pd


def clean_air_quality_data(
    input_path: str = '',
    output_path: str = "cleaned_urdaneta_air_quality.csv",
    municipality_name: str = "Urdaneta",
) -> pd.DataFrame:
    # 1. Load dataset, parsing non-numeric placeholders as NaN
    df = pd.read_csv(input_path, na_values=["-", "NA", "null", ""])

    # 2. Parse Date column
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")

    # 3. Convert air pollutant columns to numeric floats
    pollutants = ["pm2.5", "pm10", "co", "o3", "so2", "nox"]
    for col in pollutants:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4. Invalidate negative values and error codes (-999.0)
    for col in pollutants:
        df.loc[df[col] < 0, col] = np.nan

    # 5. Aggregate duplicate date entries by computing the daily mean
    df_clean = df.groupby("Date", as_index=False)[pollutants].mean()

    # 6. Ensure a continuous daily date range
    full_date_range = pd.date_range(
        start=df_clean["Date"].min(), end=df_clean["Date"].max(), freq="D"
    )
    df_clean = (
        df_clean.set_index("Date")
        .reindex(full_date_range)
        .rename_axis("Date")
        .reset_index()
    )

    # 7. Interpolate short sensor dropouts (up to 3 consecutive days)
    df_clean[pollutants] = df_clean[pollutants].interpolate(
        method="linear", limit=3, limit_direction="forward"
    )

    # 8. Add Municipality column next to Date
    df_clean.insert(1, "Municipality", municipality_name)

    # 9. Export to CSV
    df_clean.to_csv(output_path, index=False)
    print(f"Data saved to {output_path} (Shape: {df_clean.shape})")

    return df_clean


if __name__ == "__main__":
    cleaned_df = clean_air_quality_data("combined_output.csv")
    print(cleaned_df.head())